# TimesFM 2026 Forecast & Early Evaluation (Force Level)

Forecasts Jan–Mar 2026 at **force level** (total demand across all crime types) using TimesFM zero-shot, then evaluates against the 3 months of actuals we have.

**5 forces:** Cheshire, Lincolnshire, Merseyside, Metropolitan, West Midlands  
**Training context:** 2012–2019 + 2022–2025 (pandemic years excluded)  
**Forecast horizon:** Jan–Mar 2026 (3 months)  
**Evaluation:** forecast vs actuals for those 3 months

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import timesfm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

## 1. Load and aggregate to force level

In [ ]:
FORCES = {
    'Cheshire Constabulary':         'Cheshire',
    'Lincolnshire Police':            'Lincolnshire',
    'Merseyside Police':              'Merseyside',
    'Metropolitan Police Service':    'Metropolitan',
    'West Midlands Police':           'West Midlands',
}

CRIME_TYPES = [
    'Violence and sexual offences',
    'Criminal damage and arson',
    'Drugs',
    'Violent crime',
    'Burglary',
    'Other theft',
    'Vehicle crime',
    'Public order',
    'Other crime',
    'Shoplifting',
    'Robbery',
    'Bicycle theft',
    'Theft from the person',
    'Possession of weapons',
    'Public disorder and weapons',
    'Anti-social behaviour',
]

TRAIN_YEARS = set(range(2012, 2020)) | set(range(2022, 2026))
EVAL_MONTHS = pd.date_range('2026-01-01', periods=3, freq='MS')

raw = pd.read_parquet('../data/processed/crimes_clean_dedup_all_years.parquet')
raw = raw[raw['Falls within'].isin(FORCES) & raw['Crime type'].isin(CRIME_TYPES)].copy()
raw['force'] = raw['Falls within'].map(FORCES)
raw['month'] = pd.to_datetime(raw['Month'])
raw = raw[raw['month'].dt.year >= 2012]

# Aggregate all crime types → total monthly count per force
force_monthly = (
    raw.groupby(['force', 'month'])
    .size()
    .reset_index(name='count')
)

print('Forces:', force_monthly['force'].unique())
print('Date range:', force_monthly['month'].min().date(), '→', force_monthly['month'].max().date())
print('\nMonthly counts sample:')
display(force_monthly[force_monthly['month'].dt.year == 2026])

## 2. Load TimesFM

In [ ]:
print('Loading TimesFM...')
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
tfm.compile(timesfm.ForecastConfig(
    max_context=512,
    max_horizon=128,
    per_core_batch_size=32,
    infer_is_positive=True,
    normalize_inputs=True,
))
print('Ready.')

## 3. Forecast Jan–Mar 2026

In [ ]:
force_labels = list(FORCES.values())

# Build one time series per force from training data
inputs = []
for force in force_labels:
    series = (
        force_monthly[
            (force_monthly['force'] == force) &
            (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
        ]
        .sort_values('month')['count']
        .values
        .astype(float)
    )
    inputs.append(series)
    print(f'{force}: {len(series)} months of context, last value = {series[-1]:.0f}')

# Forecast 3 months
point_forecast, _ = tfm.forecast(horizon=3, inputs=inputs)
forecasts = np.clip(np.array(point_forecast)[:, :3], 0, None)

print('\nForecasts (Jan–Mar 2026):')
forecast_df = pd.DataFrame(
    forecasts,
    index=force_labels,
    columns=EVAL_MONTHS.strftime('%Y-%m')
)
display(forecast_df.round(0).astype(int))

## 4. Attach actuals and evaluate

In [ ]:
# Actuals for Jan-Mar 2026
actuals_26 = (
    force_monthly[force_monthly['month'].isin(EVAL_MONTHS)]
    .pivot(index='force', columns='month', values='count')
)
actuals_26.columns = actuals_26.columns.strftime('%Y-%m')
actuals_26 = actuals_26.reindex(force_labels)

# Baseline: mean of 2025
baseline_26 = (
    force_monthly[force_monthly['month'].dt.year == 2025]
    .groupby('force')['count'].mean()
    .reindex(force_labels)
)

print('Actuals (Jan–Mar 2026):')
display(actuals_26.round(0))

print('\n2025 monthly mean (baseline):')
display(baseline_26.round(0).to_frame('baseline_monthly_mean'))

## 5. Evaluation metrics

In [ ]:
# metric helpers (defined once, reused across the notebook)
def mae(pred, actual):
    return float(np.mean(np.abs(pred - actual)))

def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

def mape(pred, actual):
    # guard against division by zero where actual demand is 0
    return float(np.mean(np.abs((pred - actual) / np.where(actual == 0, 1, actual))) * 100)

rows = []
for i, force in enumerate(force_labels):
    actual_vals   = actuals_26.loc[force].values.astype(float)
    forecast_vals = forecasts[i]
    baseline_vals = np.full(3, baseline_26[force])

    rows.append({
        'force':         force,
        'baseline_mae':  mae(baseline_vals, actual_vals),
        'model_mae':     mae(forecast_vals,  actual_vals),
        'baseline_rmse': rmse(baseline_vals, actual_vals),
        'model_rmse':    rmse(forecast_vals,  actual_vals),
        'model_mape':    mape(forecast_vals,  actual_vals),
    })

eval_df = pd.DataFrame(rows)
eval_df['delta_mae'] = eval_df['baseline_mae'] - eval_df['model_mae']
eval_df['rmae']      = eval_df['model_mae'] / eval_df['baseline_mae']
eval_df['model_wins'] = eval_df['model_mae'] < eval_df['baseline_mae']

display(
    eval_df[['force','baseline_mae','model_mae','delta_mae','rmae','model_mape','model_wins']]
    .round(2)
    .set_index('force')
    .rename(columns={
        'baseline_mae':'Baseline MAE','model_mae':'TimesFM MAE',
        'delta_mae':'Δ MAE','rmae':'RMAE','model_mape':'MAPE %','model_wins':'Model wins'
    })
)

print(f"\nOverall: TimesFM MAE={eval_df['model_mae'].mean():.1f}  Baseline MAE={eval_df['baseline_mae'].mean():.1f}")
print(f"Win rate: {eval_df['model_wins'].mean()*100:.0f}%  |  Mean MAPE: {eval_df['model_mape'].mean():.1f}%")

## 6. Forecast vs Actual chart — per force

In [ ]:
# Include some training history for context
HISTORY_START = '2023-01-01'

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
fig.suptitle('2026 Forecast vs Actual (Jan–Mar) — Force Level', fontsize=13, fontweight='bold')

for i, (ax, force) in enumerate(zip(axes, force_labels)):
    history = force_monthly[
        (force_monthly['force'] == force) &
        (force_monthly['month'] >= HISTORY_START) &
        (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
    ].sort_values('month')

    # Plot recent history
    ax.plot(history['month'], history['count'], color='#94a3b8',
            linewidth=1.2, label='History')

    # forecast points
    fc_vals = forecasts[i]
    ax.plot(EVAL_MONTHS, fc_vals, color='#2196F3', linewidth=2,
            marker='o', markersize=6, linestyle='--', label='Forecast', zorder=5)

    # Actual points
    act_vals = actuals_26.loc[force].values
    ax.plot(EVAL_MONTHS, act_vals, color='#FF5722', linewidth=2,
            marker='o', markersize=6, label='Actual', zorder=5)

    # Baseline
    ax.axhline(baseline_26[force], color='#9E9E9E', linestyle=':',
               linewidth=1.2, label='2025 mean')

    ax.set_title(force, fontsize=10, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.tick_params(axis='y', labelsize=8)

    # Annotate MAPE
    mape = eval_df[eval_df['force'] == force]['model_mape'].values[0]
    ax.text(0.97, 0.97, f'MAPE {mape:.1f}%', transform=ax.transAxes,
            ha='right', va='top', fontsize=8,
            color='green' if mape < 10 else 'orange' if mape < 20 else 'red')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout(rect=[0, 0.05, 1, 1])
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/2026_force_level_forecast_eval.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Month-by-month error

In [ ]:
month_rows = []
for i, month in enumerate(EVAL_MONTHS):
    for fi, force in enumerate(force_labels):
        actual   = float(actuals_26.loc[force, month.strftime('%Y-%m')])
        forecast = float(forecasts[fi, i])
        baseline = float(baseline_26[force])
        month_rows.append({
            'month': month, 'force': force,
            'actual': actual, 'forecast': forecast, 'baseline': baseline,
            'forecast_error': forecast - actual,
            'baseline_error': baseline - actual,
        })

month_df = pd.DataFrame(month_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute error by month
month_mae = month_df.groupby('month').apply(
    lambda g: pd.Series({
        'TimesFM MAE': np.mean(np.abs(g['forecast_error'])),
        'Baseline MAE': np.mean(np.abs(g['baseline_error'])),
    })
).reset_index()

x = np.arange(3)
w = 0.35
axes[0].bar(x - w/2, month_mae['TimesFM MAE'], w, label='TimesFM', color='#2196F3')
axes[0].bar(x + w/2, month_mae['Baseline MAE'], w, label='Baseline', color='#9E9E9E')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Jan 2026', 'Feb 2026', 'Mar 2026'])
axes[0].set_title('MAE by Month (across 5 forces)', fontweight='bold')
axes[0].set_ylabel('MAE')
axes[0].legend()

# Forecast error per force per month (heatmap)
err_pivot = month_df.pivot_table(
    index='force', columns='month', values='forecast_error', aggfunc='mean'
)
err_pivot.columns = err_pivot.columns.strftime('%b %Y')
sns.heatmap(
    err_pivot.round(0), annot=True, fmt='.0f', cmap='RdYlGn_r',
    center=0, linewidths=0.5, ax=axes[1],
    cbar_kws={'label': 'Forecast error (+ = over, - = under)'}
)
axes[1].set_title('Forecast Error by Force × Month\n(+ = over-predicted, - = under-predicted)', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../outputs/2026_monthly_error.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save

In [ ]:
forecast_df.to_csv('../outputs/timesfm_2026_Q1_forecasts.csv')
eval_df.to_csv('../outputs/timesfm_2026_Q1_eval.csv', index=False)
print('Saved to outputs/')